# Assignment 4: Data Wrangling - Boston Housing

Clean column names, handle missing values and duplicates, encode categoricals, summarize prices, check outliers and export the prepared data.

**Dataset:** `Boston Housing Data.csv` (the Kaggle version with missing values). On Colab keep it in `MyDrive/Data-Science`; on local Jupyter keep it next to this notebook.

In [ ]:
# Mount Google Drive (run in Google Colab)
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/Data-Science'
except ImportError:  # local Jupyter
    DATA_DIR = '.'

CSV_PATH = os.path.join(DATA_DIR, 'Boston Housing Data.csv')

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

if not os.path.exists(CSV_PATH):
    print("'Boston Housing Data.csv' not found - using a public copy of the Boston data "
          "(it has NO missing values, so the missing-value outputs will differ).")
    CSV_PATH = 'https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv'

df = pd.read_csv(CSV_PATH)
print('Dataset shape:', df.shape)
display(df.head())

## 1. Clean column names and inspect data

In [ ]:
# Lowercase names, replace spaces/special characters with underscores
import re
def clean_name(name):
    name = str(name).strip().lower()
    name = re.sub(r'[^a-z0-9]+', '_', name)
    return name.strip('_')

df.columns = [clean_name(c) for c in df.columns]
print('Columns:', df.columns.tolist())
display(df.info())
display(df.describe(include='all').T)
print('Missing values:')
display(df.isna().sum().to_frame('missing_count'))
print('Duplicate rows:', df.duplicated().sum())

## 2. Handle missing values and duplicates

Remove duplicate records. Fill numeric gaps with the column median and categorical gaps with the most common value.

In [ ]:
df = df.drop_duplicates().copy()
for col in df.columns:
    if df[col].isna().any():
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            mode = df[col].mode(dropna=True)
            df[col] = df[col].fillna(mode.iloc[0] if not mode.empty else 'Unknown')
print('Remaining missing cells:', int(df.isna().sum().sum()))

## 3. Filter rows and encode categorical columns

No location or date filter is applied by default because datasets differ. Edit the optional example after reviewing your columns. Categorical columns are one-hot encoded for analysis.

In [ ]:
# Optional example: filter if your dataset has a suitable column.
# Uncomment and change the column/value after checking df.columns:
# df = df[df['neighborhood'] == 'your neighborhood']

categorical_cols = df.select_dtypes(include=['object','string','category']).columns.tolist()
print('Categorical columns:', categorical_cols)
encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)
print('Encoded dataset shape:', encoded.shape)
display(encoded.head())

## 4. Summary statistics and derived metrics

If a likely house-price column is present, calculate its average by categorical columns. Column names are detected flexibly; review the detected result.

In [ ]:
price_candidates = [c for c in df.columns if any(word in c for word in ['price', 'value', 'medv'])]
print('Possible price columns:', price_candidates)
if price_candidates:
    price_col = price_candidates[0]
    print('Using price column:', price_col)
    print('Average price/value:', df[price_col].mean())
    for col in categorical_cols:
        summary = df.groupby(col, dropna=False)[price_col].agg(['count','mean'])
        print(f'Average {price_col} by {col}:')
        display(summary)
else:
    print('No obvious price column found. Review the columns and set price_col manually.')
display(df.describe(include='all').T)

## 5. Find and handle numeric outliers

Use the 1.5×IQR rule to flag outliers. Keep them by default because unusual values may be valid. The code also creates a copy with outlier rows removed if you explicitly want it.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
outlier_counts = {}
for col in numeric_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    mask = (df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)
    outlier_counts[col] = int(mask.sum())
display(pd.Series(outlier_counts, name='IQR outlier count').to_frame())

# Optional: remove rows with an IQR outlier in any numeric column.
# Uncomment to create a filtered copy, while keeping `df` unchanged:
# keep = pd.Series(True, index=df.index)
# for col in numeric_cols:
#     q1, q3 = df[col].quantile([0.25, 0.75]); iqr = q3 - q1
#     keep &= df[col].between(q1 - 1.5*iqr, q3 + 1.5*iqr)
# df_no_outliers = df.loc[keep].copy()

## 6. Export cleaned and encoded data

In [ ]:
OUT_DIR = Path(DATA_DIR) / 'boston_housing_prepared'
OUT_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_DIR / 'housing_cleaned.csv', index=False)
encoded.to_csv(OUT_DIR / 'housing_encoded.csv', index=False)
print('Saved cleaned and encoded files to:', OUT_DIR)